# PAP_NER Benchmark — Google Colab

Đồng bộ với `run_kaggle.ipynb` (paper-like PLM+CRF, multi-seed, bamibert, Macro-F1, error analysis ĐT).

**Baseline paper:** PhoBERT-CRF Micro-F1 = **97.95%**

**Chuẩn bị Colab**
1. Runtime → Change runtime type → **GPU** (T4 / L4 / A100). Tránh CPU.
2. Upload / mount thư mục `src` (có `train.py`, `config.py`, ideally `data/pap_ner/*.txt`).
3. Chạy **từng step**; cuối step phải có `[DONE]` hoặc `[ERROR]`.
4. Colab dễ disconnect: bật **Drive** để lưu `outputs/`, hoặc Download sau mỗi model.

**Step**
- **1** Setup path + deps (pin `transformers==4.46.3`)
- **2** Data PAP_NER
- **3** Smoke test
- **4.0** Shared config + helpers
- **4.1** `phobert` · **4.1A** multi-seed · **4.2** `phobert_v2` · **4.3** `xlmr` · **4.4** `bamibert`
- **5** Results (Micro/Macro, mean±std)
- **6** Error analysis ĐT


### (Tuỳ chọn) Giải nén `pap-ner-src.zip` trên `/content`

Chạy nếu bạn Upload zip vào Files sidebar. Đổi tên file nếu khác.


In [ ]:
STEP = "Step 1b Unzip"
try:
    import zipfile
    from pathlib import Path
    zip_path = Path("/content/pap-ner-src.zip")
    # fallback: bất kỳ zip chứa src
    if not zip_path.exists():
        cands = list(Path("/content").glob("*.zip"))
        zip_path = cands[0] if cands else zip_path
    assert zip_path.exists(), f"Không thấy zip tại {zip_path}"
    out = Path("/content/pap-ner-src")
    out.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out)
    print("Extracted to", out)
    print("sample:", sorted(p.name for p in out.rglob("train.py"))[:3])
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 1 — Setup path + dependencies (Colab)

Ưu tiên (chọn 1):
1. **Google Drive:** bỏ `src` vào Drive rồi mount (ổn định khi reconnect).
2. **Upload zip** `pap-ner-src.zip` lên `/content` rồi giải nén.
3. Clone / copy tay vào `/content/src`.

Cell này copy code → `/content/pap_ner_src` (writable) và pin transformers.


In [ ]:
STEP = "Step 1 Setup"
try:
    import os, sys, subprocess, shutil
    from pathlib import Path

    # --- optional: mount Drive ---
    MOUNT_DRIVE = True  # False nếu chỉ upload zip /content
    DRIVE_SRC_CANDIDATES = [
        Path("/content/drive/MyDrive/ThS_HUIT/PhuongPhapNghienCuuKhoaHoc/DieuTra/src"),
        Path("/content/drive/MyDrive/PAP_NER/src"),
        Path("/content/drive/MyDrive/pap-ner-src/src"),
        Path("/content/drive/MyDrive/src"),
    ]

    if MOUNT_DRIVE and Path("/content").exists():
        try:
            from google.colab import drive  # type: ignore
            drive.mount("/content/drive", force_remount=False)
            print("Drive mounted")
        except Exception as e:
            print("Drive mount skipped:", type(e).__name__, e)

    def looks_like_src(p: Path) -> bool:
        return (p / "train.py").exists() and (p / "config.py").exists()

    candidates = []
    # zip đã giải nén
    for z in [
        Path("/content/pap-ner-src/src"),
        Path("/content/pap-ner-src"),
        Path("/content/src"),
    ]:
        candidates.append(z)
    candidates.extend(DRIVE_SRC_CANDIDATES)
    candidates += [
        Path("/kaggle/input/pap-ner-src/src"),
        Path("/kaggle/working/src"),
        Path.cwd() / "src",
        Path.cwd(),
    ]
    # search Drive shallow for config.py (optional, capped)
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        found = 0
        for cfg in drive_root.rglob("config.py"):
            parent = cfg.parent
            if looks_like_src(parent):
                candidates.insert(0, parent)
                found += 1
            if found >= 5:
                break

    src_in = next((p for p in candidates if p.exists() and looks_like_src(p)), None)
    assert src_in is not None, (
        "Chưa thấy src. Upload pap-ner-src.zip → /content rồi giải nén, "
        "hoặc đặt SRC trên Drive và set DRIVE_SRC_CANDIDATES."
    )

    # Copy sang writable location (tránh Drive I/O chậm khi train)
    SRC_DIR = Path("/content/pap_ner_src")
    if SRC_DIR.exists():
        shutil.rmtree(SRC_DIR)
    shutil.copytree(src_in, SRC_DIR, dirs_exist_ok=True)

    sys.path = [str(SRC_DIR)] + [p for p in sys.path if p != str(SRC_DIR)]
    os.chdir(SRC_DIR)
    print("SRC_IN  =", src_in)
    print("SRC_DIR =", SRC_DIR)

    def pip_install(pkgs: list[str]) -> None:
        """Install pkgs; on failure print stderr (Colab -q che lỗi)."""
        cmd = [sys.executable, "-m", "pip", "install", "--upgrade", *pkgs]
        print(">>", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True)
        if proc.returncode != 0:
            print("--- pip stdout (tail) ---")
            print((proc.stdout or "")[-2500:])
            print("--- pip stderr (tail) ---")
            print((proc.stderr or "")[-3500:])
            raise RuntimeError(f"pip failed for {pkgs} (exit {proc.returncode})")
        print("OK:", ", ".join(pkgs))

    # Python 3.13 (Colab mới): 4.46.3 thường không có wheel → dùng 4.48+ (vẫn <5)
    # Python ≤3.12 (Kaggle/Colab cũ): pin 4.46.3 như Kaggle
    py = sys.version_info
    print(f"Python {py.major}.{py.minor}.{py.micro}")
    if py >= (3, 13):
        TRANSFORMERS_SPEC = "transformers>=4.48.0,<5.0"
        print("Py≥3.13 →", TRANSFORMERS_SPEC)
    else:
        TRANSFORMERS_SPEC = "transformers==4.46.3"
        print("Py≤3.12 →", TRANSFORMERS_SPEC)

    print("Installing deps ...")
    # Cài từng gói: 1 gói fail không chặn cả stack.
    # seqeval không hỗ trợ tốt Py3.13 (setup.py egg_info) — project dùng metrics.py riêng, bỏ qua.
    pip_install([TRANSFORMERS_SPEC])
    for pkg in ("accelerate", "pandas"):
        try:
            pip_install([pkg])
        except Exception as e:
            print(f"WARNING: optional skip {pkg}: {e}")
    try:
        pip_install(["sentencepiece>=0.1.99"])
    except Exception as e:
        print("WARNING: sentencepiece install failed — PhoBERT/XLM-R thường vẫn chạy:", e)
    print("Note: skip seqeval (Py3.13); dùng metrics.py trong src.")

    req = SRC_DIR / "requirements.txt"
    if req.exists():
        # Tránh -r kéo lại transformers==4.46.3 trên Py3.13
        try:
            filtered = SRC_DIR / "_requirements_colab.txt"
            lines = []
            for line in req.read_text().splitlines():
                low = line.strip().lower()
                if low.startswith("transformers"):
                    continue
                if low.startswith("sentencepiece") and py >= (3, 13):
                    continue
                lines.append(line)
            filtered.write_text("\n".join(lines) + "\n")
            pip_install(["-r", str(filtered)])
        except Exception as e:
            print("WARNING: requirements.txt partial fail:", e)

    for m in list(sys.modules):
        if m == "transformers" or m.startswith("transformers."):
            del sys.modules[m]

    import torch
    import transformers
    from crf import CRF

    print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
    print("transformers", transformers.__version__)
    assert transformers.__version__.startswith("4."), (
        f"Cần transformers 4.x, đang {transformers.__version__}. "
        "Gỡ 5.x: !pip install -U 'transformers>=4.48,<5'"
    )

    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} | sm_{major}{minor} | VRAM≈{vram:.1f}GB")
        x = torch.randn(2, 2, device="cuda")
        print("CUDA OK", float((x @ x).sum()))
    else:
        print("WARNING: CPU only — full-data sẽ rất chậm. Đổi Runtime → GPU.")

    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 2 — Download PAP_NER


In [ ]:
STEP = "Step 2 Prepare data"
try:
    from utils import default_paths, environment_report
    from download_data import ensure_pap_ner_data

    print(environment_report())
    PATHS = default_paths()
    for k, v in PATHS.items():
        v.mkdir(parents=True, exist_ok=True)
        print(f"{k}: {v}")

    DATA_DIR = PATHS["data"]
    OUTPUT_DIR = PATHS["outputs"]

    # Ưu tiên data đóng gói trong zip; chỉ Zenodo nếu thiếu
    ensure_pap_ner_data(DATA_DIR, allow_download=True)

    files = sorted(DATA_DIR.glob("*.txt"))
    assert len(files) >= 3, f"Thiếu file data: {files}"
    for f in files:
        print(f"  {f.name}: {f.stat().st_size:,} bytes")
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise



## Step 3 — Smoke test

Chạy nhanh 1 epoch subset. Sau pipeline polish:
- **`train_loss` phải ≥ 0** (NLL)
- Metric word-level (first-subword)
- Train không gọi Viterbi mỗi step (nhanh hơn)

Tuỳ chọn CLI: `python smoke_crf_loss.py`


In [ ]:
STEP = "Step 3 Smoke test"
try:
    from config import TrainConfig
    from train import train

    smoke_cfg = TrainConfig(
        model_key="phobert",
        use_crf=True,
        data_dir=str(DATA_DIR),
        output_dir=str(OUTPUT_DIR / "smoke"),
        num_epochs=1,
        batch_size=8,
        eval_batch_size=16,
        max_train_samples=2000,
        max_eval_samples=500,
        max_test_samples=500,
        seed=42,
        num_workers=0,
    )
    smoke_result = train(smoke_cfg)
    f1 = smoke_result["test"]["overall"]["f1"] * 100
    print(f"Smoke Micro-F1 = {f1:.2f}")
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 4.0 — Cấu hình chung (paper-like)

Giữ **CRF + hyperparams Table 4**; chỉ đổi PLM. Chạy cell này **một lần** trước 4.1 / 4.1A / 4.2 / 4.3.

| Hạng mục | Giá trị |
|---|---|
| Data | Full |
| Encoder / classifier / CRF LR | 2e-5 / 1e-3 / 1e-3 |
| Batch | 32 (`xlmr` → 16) |
| Max epochs | 20 |
| Early stop (baseline) | patience **3**, min_delta 1e-4 |
| Early stop (nhánh A) | `OPTIMIZE_A=True` → patience **5** |
| Seeds | `SEED=42`; multi-seed `SEEDS=[42,123,2024]` |

Helpers: `run_one_model(...)`, `run_multi_seed(...)`.

Model keys: `phobert` | `phobert_v2` | `xlmr` | **`bamibert`**.

`bamibert` (`Qualcomm-AI-Research/BamiBERT`): batch ≤16; `BAMIBERT_MAX_LENGTH=256` mặc định (công bằng vs PhoBERT). Đổi 512 trong 4.0 nếu muốn probe span dài.

Mỗi model × 1 seed ~2–5 giờ trên T4 (`bamibert` ~2.5–3.5h @256). Multi-seed ≈ ×3 — chạy từng model riêng.

Trên Colab: bật `SYNC_TO_DRIVE=True` ở cell code 4.0 để tự copy `best.pt` / `results.json` sang Drive (tránh mất khi disconnect).


In [ ]:
STEP = "Step 4.0 Shared config"
try:
    from pathlib import Path
    import shutil
    from utils import save_json
    from config import TrainConfig, paper_like_batch_size, paper_like_max_length
    from train import train

    PAPER_LIKE = True
    FULL_DATA = True if PAPER_LIKE else False
    NUM_EPOCHS = 20 if PAPER_LIKE else 3
    BATCH_SIZE = 32 if PAPER_LIKE else 16
    MAX_LENGTH = 256  # Table 4 / fair vs PhoBERT
    # BamiBERT hỗ trợ tới 2048; giữ 256 để so công bằng. Đổi 512 nếu probe ĐT dài.
    BAMIBERT_MAX_LENGTH = 256
    # Nhánh A: patience lớn hơn để train gần ep 15–17 như paper
    OPTIMIZE_A = True
    EARLY_STOPPING = (5 if OPTIMIZE_A else 3) if PAPER_LIKE else 0
    EARLY_STOPPING_MIN_DELTA = 1e-4
    LEARNING_RATE = 2e-5
    CLASSIFIER_LEARNING_RATE = 1e-3
    CRF_LEARNING_RATE = 1e-3
    SEED = 42
    SEEDS = [42, 123, 2024]  # multi-seed như paper (mean ± std)
    WITH_ABLATION = False  # True = thêm Softmax (không bắt buộc)

    max_train = -1 if FULL_DATA else 20000
    max_eval = -1 if FULL_DATA else 2000
    max_test = -1 if FULL_DATA else 2000

    if "summary" not in globals() or summary is None:
        summary = []

    def _macro_f1_from_test(test: dict):
        """Macro-F1 = mean F1 over entity types (paper Table 7/9)."""
        if not test:
            return None
        macro = test.get("macro") or {}
        if macro.get("f1") is not None:
            return float(macro["f1"])
        pe = test.get("per_entity") or {}
        if not pe:
            return None
        return sum(float(v["f1"]) for v in pe.values()) / len(pe)

    def run_one_model(
        model_key: str,
        use_crf: bool = True,
        seed: int = SEED,
        early_stopping_patience=None,
        max_length=None,
    ):
        bs = paper_like_batch_size(model_key, BATCH_SIZE)
        patience = (
            EARLY_STOPPING
            if early_stopping_patience is None
            else early_stopping_patience
        )
        if max_length is None:
            if model_key == "bamibert":
                max_length = BAMIBERT_MAX_LENGTH
            else:
                max_length = paper_like_max_length(model_key, MAX_LENGTH)
        cfg = TrainConfig(
            model_key=model_key,
            use_crf=use_crf,
            data_dir=str(DATA_DIR),
            output_dir=str(OUTPUT_DIR),
            num_epochs=NUM_EPOCHS,
            batch_size=bs,
            max_length=max_length,
            learning_rate=LEARNING_RATE,
            classifier_learning_rate=CLASSIFIER_LEARNING_RATE,
            crf_learning_rate=CRF_LEARNING_RATE,
            early_stopping_patience=patience,
            early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
            max_train_samples=max_train,
            max_eval_samples=max_eval,
            max_test_samples=max_test,
            seed=seed,
            num_workers=0,
        )
        print(
            "\n===",
            model_key,
            "CRF=",
            use_crf,
            "seed=",
            seed,
            "bs=",
            bs,
            "max_len=",
            max_length,
            "patience=",
            patience,
            "===",
        )
        result = train(cfg)
        row = {
            "model_key": model_key,
            "use_crf": use_crf,
            "seed": seed,
            "batch_size": bs,
            "early_stopping_patience": patience,
            "best_epoch": result.get("best_epoch"),
            "stopped_early": result.get("stopped_early"),
            "test_micro_f1": result["test"]["overall"]["f1"],
            "test_macro_f1": _macro_f1_from_test(result["test"]),
            "per_entity": result["test"]["per_entity"],
            "train_time_sec": result["train_time_sec"],
            "peak_gpu_memory_mb": result["peak_gpu_memory_mb"],
            "num_parameters": result["num_parameters"],
        }
        global summary
        summary = [
            r
            for r in summary
            if not (
                r.get("model_key") == model_key
                and r.get("seed") == seed
                and r.get("use_crf") == use_crf
            )
        ]
        summary.append(row)
        save_json(summary, OUTPUT_DIR / "benchmark_summary.json")
        print(
            f"[DONE] model={model_key} crf={use_crf} seed={seed} "
            f"micro={row['test_micro_f1']*100:.2f} "
            f"macro={(row['test_macro_f1'] or 0)*100:.2f} "
            f"best_ep={row['best_epoch']}"
        )
        print("Saved", OUTPUT_DIR / "benchmark_summary.json")
        print("Checkpoint dir:", cfg.run_dir())
        return row

    def run_multi_seed(
        model_key: str,
        seeds=None,
        use_crf: bool = True,
        early_stopping_patience=None,
        skip_existing: bool = True,
    ):
        """Train nhiều seed; bỏ qua seed đã có results.json nếu skip_existing."""
        seeds = seeds or SEEDS
        rows = []
        for seed in seeds:
            run_dir = (
                Path(OUTPUT_DIR)
                / f"{model_key}_{'crf' if use_crf else 'softmax'}_seed{seed}"
            )
            results_path = run_dir / "results.json"
            if skip_existing and results_path.exists():
                import json

                r = json.loads(results_path.read_text())
                print(
                    f"[SKIP] {results_path} already exists — "
                    f"micro={r['test']['overall']['f1']*100:.2f}"
                )
                row = {
                    "model_key": model_key,
                    "use_crf": use_crf,
                    "seed": seed,
                    "batch_size": paper_like_batch_size(model_key, BATCH_SIZE),
                    "early_stopping_patience": early_stopping_patience
                    if early_stopping_patience is not None
                    else EARLY_STOPPING,
                    "best_epoch": r.get("best_epoch"),
                    "stopped_early": r.get("stopped_early"),
                    "test_micro_f1": r["test"]["overall"]["f1"],
                    "test_macro_f1": _macro_f1_from_test(r["test"]),
                    "per_entity": r["test"].get("per_entity"),
                    "train_time_sec": r.get("train_time_sec"),
                    "peak_gpu_memory_mb": r.get("peak_gpu_memory_mb"),
                    "num_parameters": r.get("num_parameters"),
                    "skipped": True,
                }
                global summary
                summary = [
                    x
                    for x in summary
                    if not (
                        x.get("model_key") == model_key
                        and x.get("seed") == seed
                        and x.get("use_crf") == use_crf
                    )
                ]
                summary.append(row)
                save_json(summary, OUTPUT_DIR / "benchmark_summary.json")
                rows.append(row)
                continue
            rows.append(
                run_one_model(
                    model_key,
                    use_crf=use_crf,
                    seed=seed,
                    early_stopping_patience=early_stopping_patience,
                )
            )
        return rows

    print(
        f"Protocol={'paper_like' if PAPER_LIKE else 'subset'} | FULL_DATA={FULL_DATA} | "
        f"epochs<={NUM_EPOCHS} | patience={EARLY_STOPPING} (OPTIMIZE_A={OPTIMIZE_A}) | "
        f"SEED={SEED} | SEEDS={SEEDS}"
    )
    print(
        "Helpers: run_one_model('phobert'|'phobert_v2'|'xlmr'|'bamibert'), "
        "run_multi_seed('phobert')"
    )
    print(f"BAMIBERT_MAX_LENGTH={BAMIBERT_MAX_LENGTH} (batch bamibert≤16)")
    

    # Colab: optionally mirror outputs to Drive after each run
    DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/PAP_NER/outputs")  # đổi nếu cần
    SYNC_TO_DRIVE = True  # False nếu chưa mount Drive

    def sync_run_to_drive(run_dir):
        """Copy một run_dir (best.pt, results.json, ...) sang Drive."""
        if not SYNC_TO_DRIVE:
            return
        run_dir = Path(run_dir)
        if not run_dir.exists():
            print("[drive] skip, missing", run_dir)
            return
        if not Path("/content/drive/MyDrive").exists():
            print("[drive] MyDrive chưa mount — bỏ qua sync")
            return
        dest = DRIVE_OUTPUT_DIR / run_dir.name
        dest.mkdir(parents=True, exist_ok=True)
        for name in ("results.json", "run_meta.json", "best.pt", "tokenizer"):
            src_p = run_dir / name
            if not src_p.exists():
                continue
            dst_p = dest / name
            if src_p.is_dir():
                if dst_p.exists():
                    shutil.rmtree(dst_p)
                shutil.copytree(src_p, dst_p)
            else:
                shutil.copy2(src_p, dst_p)
        print("[drive] synced", run_dir, "->", dest)

    _orig_run_one_model = run_one_model

    def run_one_model(*args, **kwargs):
        row = _orig_run_one_model(*args, **kwargs)
        try:
            from config import TrainConfig
            # reconstruct run dir from last train — use OUTPUT_DIR pattern
            key = kwargs.get("model_key", args[0] if args else None)
            use_crf = kwargs.get("use_crf", True)
            seed = kwargs.get("seed", SEED)
            tag = "crf" if use_crf else "softmax"
            sync_run_to_drive(Path(OUTPUT_DIR) / f"{key}_{tag}_seed{seed}")
        except Exception as e:
            print("[drive] sync warning:", e)
        return row

    print("Colab sync helper: sync_run_to_drive / run_one_model wraps Drive copy")
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 4.1 — Train `phobert` + CRF (1 seed)

Baseline nhanh với `SEED` (mặc định 42). Dùng patience từ Step 4.0 (`OPTIMIZE_A` → 5).

Sau `[DONE]`, Download / copy Drive `outputs/phobert_crf_seed42/`. Muốn mean±std như paper → chạy **4.1A**.


In [ ]:
STEP = "Step 4.1 phobert"
try:
    assert "run_one_model" in globals(), "Chạy Step 4.0 trước"
    row = run_one_model("phobert", use_crf=True)
    if WITH_ABLATION:
        run_one_model("phobert", use_crf=False)
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 4.1A — `phobert` multi-seed (nhánh A)

Train `SEEDS=[42, 123, 2024]` với patience lớn hơn (từ 4.0).  

- `skip_existing=True`: bỏ qua seed đã có `results.json` (tiếp tục session / upload checkpoint cũ).  
- Thời gian ≈ 3 × Step 4.1 — nên session riêng, Download từng seed.  
- Step 5 sẽ in **mean ± std** Micro-F1.


In [ ]:
STEP = "Step 4.1A phobert multi-seed"
try:
    assert "run_multi_seed" in globals(), "Chạy Step 4.0 trước"
    rows = run_multi_seed(
        "phobert",
        seeds=SEEDS,
        use_crf=True,
        skip_existing=True,
    )
    f1s = [r["test_micro_f1"] * 100 for r in rows]
    if len(f1s) >= 2:
        import statistics as stats

        mean = stats.mean(f1s)
        std = stats.pstdev(f1s) if len(f1s) > 1 else 0.0
        print(
            f"phobert multi-seed Micro-F1 = {mean:.2f} ± {std:.2f} "
            f"(n={len(f1s)}, vs paper 97.95 → Δ={mean - 97.95:+.2f})"
        )
    for r in rows:
        print(
            f"  seed={r['seed']}: {r['test_micro_f1']*100:.2f} "
            f"best_ep={r.get('best_epoch')} skipped={r.get('skipped', False)}"
        )
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 4.2 — Train `phobert_v2` + CRF (full)

Model mới (ngoài paper). Có thể chạy session riêng sau khi đã xong 4.0 (+ 4.1 nếu cần so sánh).


In [ ]:
STEP = "Step 4.2 phobert_v2"
try:
    assert "run_one_model" in globals(), "Chạy Step 4.0 trước"
    row = run_one_model("phobert_v2", use_crf=True)
    if WITH_ABLATION:
        run_one_model("phobert_v2", use_crf=False)
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 4.3 — Train `xlmr` + CRF (full)

XLM-R nặng hơn (batch 16). Nên chạy session riêng.


In [ ]:
STEP = "Step 4.3 xlmr"
try:
    assert "run_one_model" in globals(), "Chạy Step 4.0 trước"
    row = run_one_model("xlmr", use_crf=True)
    if WITH_ABLATION:
        run_one_model("xlmr", use_crf=False)
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 4.4 — Train `bamibert` + CRF (full)

Encoder mới ngoài paper: [`Qualcomm-AI-Research/BamiBERT`](https://huggingface.co/Qualcomm-AI-Research/BamiBERT) (~100M, context tới 2048, không bắt buộc word-seg).

Trên PAP_NER vẫn **align word→subword** (`encode_example`) để metric word-level công bằng với PhoBERT.

| Hạng mục | Giá trị |
|---|---|
| Batch | ≤16 (T4) |
| `max_length` | `BAMIBERT_MAX_LENGTH` (mặc định **256**; đổi **512** ở 4.0 nếu probe ĐT dài) |
| Thời gian ước lượng | ~2.5–3.5 giờ / seed @256 |

Cần **Internet On** lần đầu để tải model. License: BSD-3-Clear + Qualcomm RAIL (research OK).

Sau `[DONE]`, Download / copy Drive `outputs/bamibert_crf_seed42/`.


In [ ]:
STEP = "Step 4.4 bamibert"
try:
    assert "run_one_model" in globals(), "Chạy Step 4.0 trước"
    # max_length lấy từ BAMIBERT_MAX_LENGTH trong 4.0
    row = run_one_model("bamibert", use_crf=True)
    print(
        f"bamibert Micro-F1={row['test_micro_f1']*100:.2f} "
        f"Macro-F1={(row.get('test_macro_f1') or 0)*100:.2f} "
        f"Δmicro={row['test_micro_f1']*100 - 97.95:+.2f}"
    )
    if WITH_ABLATION:
        run_one_model("bamibert", use_crf=False)
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 5 — Results table

Gộp kết quả từ `summary` **hoặc** `results.json` trên disk.  

In thêm **mean ± std** theo `(model, CRF)` khi có ≥2 seeds (so với paper 97.95).


In [ ]:
STEP = "Step 5 Results"
try:
    import json
    import statistics as stats
    from pathlib import Path
    import pandas as pd
    from utils import save_json

    PAPER_BASELINE = 97.95
    PAPER_MACRO = 98.22
    rows_data = []

    if "summary" in globals() and summary:
        rows_data = list(summary)
        print(f"Loaded {len(rows_data)} row(s) from in-memory summary")

    summary_path = Path(OUTPUT_DIR) / "benchmark_summary.json"
    if not rows_data and summary_path.exists():
        rows_data = json.loads(summary_path.read_text())
        print(f"Loaded {len(rows_data)} row(s) from {summary_path}")

    if not rows_data:
        for results_path in sorted(Path(OUTPUT_DIR).glob("*_seed*/results.json")):
            r = json.loads(results_path.read_text())
            meta_path = results_path.parent / "run_meta.json"
            meta = json.loads(meta_path.read_text()) if meta_path.exists() else {}
            cfg = meta.get("config", {})
            rows_data.append({
                "model_key": cfg.get("model_key", results_path.parent.name.split("_")[0]),
                "use_crf": cfg.get("use_crf", "crf" in results_path.parent.name),
                "seed": cfg.get("seed", 42),
                "batch_size": cfg.get("batch_size"),
                "best_epoch": r.get("best_epoch"),
                "stopped_early": r.get("stopped_early"),
                "test_micro_f1": r["test"]["overall"]["f1"],
                "test_macro_f1": (
                    (r["test"].get("macro") or {}).get("f1")
                    if r["test"].get("macro")
                    else (
                        sum(v["f1"] for v in (r["test"].get("per_entity") or {}).values())
                        / max(len(r["test"].get("per_entity") or {}), 1)
                        if r["test"].get("per_entity")
                        else None
                    )
                ),
                "per_entity": r["test"].get("per_entity"),
                "train_time_sec": r.get("train_time_sec"),
                "peak_gpu_memory_mb": r.get("peak_gpu_memory_mb"),
                "num_parameters": r.get("num_parameters"),
            })
        print(f"Loaded {len(rows_data)} row(s) from results.json files")

    assert rows_data, (
        f"Không có kết quả. Chạy Step 4.1–4.3 hoặc kiểm tra {OUTPUT_DIR}"
    )

    save_json(rows_data, summary_path)
    summary = rows_data  # noqa: F841

    rows = []
    for r in rows_data:
        ent = r.get("per_entity") or {}
        dt = ent.get("ĐT") or ent.get("DT") or {}
        macro = r.get("test_macro_f1")
        if macro is None and ent:
            macro = sum(float(v["f1"]) for v in ent.values()) / len(ent)
        rows.append({
            "model": r["model_key"],
            "CRF": r["use_crf"],
            "seed": r.get("seed", 42),
            "best_ep": r.get("best_epoch"),
            "Micro-F1": round(r["test_micro_f1"] * 100, 2),
            "Macro-F1": None if macro is None else round(float(macro) * 100, 2),
            "Δmicro": round(r["test_micro_f1"] * 100 - PAPER_BASELINE, 2),
            "Δmacro": None if macro is None else round(float(macro) * 100 - PAPER_MACRO, 2),
            "ĐT_F1": None if not dt else round(float(dt.get("f1", 0)) * 100, 2),
            "params_M": round((r.get("num_parameters") or 0) / 1e6, 1),
            "peak_vram_MB": None if r.get("peak_gpu_memory_mb") is None else round(r["peak_gpu_memory_mb"], 1),
            "train_min": None if r.get("train_time_sec") is None else round(r["train_time_sec"] / 60, 1),
        })
    df = pd.DataFrame(rows).sort_values("Micro-F1", ascending=False)
    display(df)

    # mean ± std theo model+CRF
    agg_rows = []
    for (model, crf), g in df.groupby(["model", "CRF"]):
        vals = g["Micro-F1"].tolist()
        macros = [x for x in g["Macro-F1"].tolist() if x is not None]
        mean = float(stats.mean(vals))
        std = float(stats.pstdev(vals)) if len(vals) > 1 else 0.0
        agg_rows.append({
            "model": model,
            "CRF": crf,
            "n_seeds": len(vals),
            "Micro-F1_mean": round(mean, 2),
            "Micro-F1_std": round(std, 2),
            "Macro-F1_mean": None if not macros else round(float(stats.mean(macros)), 2),
            "delta_micro_vs_paper": round(mean - PAPER_BASELINE, 2),
            "delta_macro_vs_paper": None
            if not macros
            else round(float(stats.mean(macros)) - PAPER_MACRO, 2),
            "best_seed_F1": round(max(vals), 2),
        })
    df_agg = pd.DataFrame(agg_rows).sort_values("Micro-F1_mean", ascending=False)
    print("\n--- Aggregate (mean ± std) | paper Micro 97.95 / Macro 98.22 ---")
    display(df_agg)

    print("Best single run:", df.iloc[0]["model"], "seed=", df.iloc[0]["seed"],
          "Micro-F1=", df.iloc[0]["Micro-F1"], "Macro-F1=", df.iloc[0]["Macro-F1"])
    print("Saved", summary_path)
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Step 6 — Error analysis ĐT (nhánh B)

Phân tích lỗi entity **ĐT** trên checkpoint đã train (không cần re-train).  

Cần: `DATA_DIR`, và `best.pt` (mặc định tìm `outputs/phobert_crf_seed42/best.pt`).  

Output: `outputs/error_analysis_DT.json` + vài ví dụ missing / boundary / type.  
Dùng kết quả để chọn ablation tiếp (`max_length`, class weight, …).


In [ ]:
STEP = "Step 6 Error analysis ĐT"
try:
    from pathlib import Path
    import torch

    from evaluate import load_checkpoint
    from data import load_pap_ner_splits, encode_example, collate_batch
    from train import split_batch_for_model, move_batch
    from utils import get_device, save_json
    from error_analysis import (
        analyze_sentence,
        summarize_error_types,
        pick_examples_by_category,
        long_entity_errors,
        span_length_histogram,
    )

    FOCUS_TYPE = "ĐT"
    # Đổi path nếu dùng seed khác / upload từ local
    CKPT = Path(OUTPUT_DIR) / "phobert_crf_seed42" / "best.pt"
    if not CKPT.exists():
        # fallback: bất kỳ phobert_* /best.pt
        cands = sorted(Path(OUTPUT_DIR).glob("phobert*_seed*/best.pt"))
        assert cands, f"Không thấy best.pt dưới {OUTPUT_DIR}. Chạy 4.1 hoặc upload checkpoint."
        CKPT = cands[0]
        print("Using fallback checkpoint:", CKPT)

    device = get_device()
    model, tokenizer, ckpt, cfg = load_checkpoint(CKPT, device)
    id2label = {int(k): v for k, v in ckpt["id2label"].items()}
    label2id = ckpt["label2id"]
    max_length = getattr(cfg, "max_length", 256)

    splits = load_pap_ner_splits(
        DATA_DIR,
        max_train_samples=1,
        max_eval_samples=1,
        max_test_samples=-1,
    )
    examples = splits["test"]
    print(f"Analyzing {len(examples):,} test sentences | focus={FOCUS_TYPE} | ckpt={CKPT}")

    analyses = []
    n_trunc = 0
    model.eval()
    bs = 16
    # Encode in batches but keep sentence metadata
    for start in range(0, len(examples), bs):
        batch_ex = examples[start : start + bs]
        feats = []
        meta = []
        for ex in batch_ex:
            feat = encode_example(
                ex.tokens, ex.labels, tokenizer, label2id, max_length=max_length
            )
            # truncated if not all words got a word_start
            n_words_enc = int(feat["word_starts"].sum().item())
            truncated = n_words_enc < len(ex.tokens)
            if truncated:
                n_trunc += 1
            feats.append(feat)
            meta.append((ex.tokens, truncated))
        batch = collate_batch(feats, pad_token_id=tokenizer.pad_token_id)
        batch = move_batch(batch, device)
        model_batch, word_starts = split_batch_for_model(batch)
        with torch.no_grad():
            out = model(**model_batch, decode=True)
        labels = model_batch["labels"].detach().cpu().tolist()
        preds = out["predictions"].detach().cpu().tolist()
        ws = word_starts.detach().cpu().tolist()
        for i, (tokens, truncated) in enumerate(meta):
            analyses.append(
                analyze_sentence(
                    tokens,
                    labels[i],
                    preds[i],
                    id2label,
                    word_starts=ws[i],
                    focus_type=FOCUS_TYPE,
                    truncated=truncated,
                )
            )
        if (start // bs) % 50 == 0:
            print(f"  … {min(start + bs, len(examples))}/{len(examples)}")

    summary_counts = summarize_error_types(analyses)
    hist_missing = span_length_histogram(analyses, "missing")
    hist_spurious = span_length_histogram(analyses, "spurious")
    long_miss = long_entity_errors(analyses, min_tokens=8)

    report = {
        "checkpoint": str(CKPT),
        "focus_type": FOCUS_TYPE,
        "n_sentences": len(analyses),
        "n_truncated_sentences": n_trunc,
        "counts": summary_counts,
        "missing_length_hist": hist_missing,
        "spurious_length_hist": hist_spurious,
        "n_sentences_with_long_missing_ĐT": len(long_miss),
    }
    out_path = Path(OUTPUT_DIR) / "error_analysis_DT.json"
    save_json(report, out_path)

    print("\n=== ĐT error buckets ===")
    for k, v in summary_counts.items():
        print(f"  {k}: {v}")
    print("Missing ĐT length hist:", hist_missing)
    print("Spurious ĐT length hist:", hist_spurious)
    print(f"Sentences truncated by max_length={max_length}: {n_trunc}")
    print(f"Sentences with long missing ĐT (≥8 tokens): {len(long_miss)}")

    def _show(cat: str, limit: int = 3):
        exs = pick_examples_by_category(analyses, cat, limit=limit)
        print(f"\n--- Examples: {cat} (up to {limit}) ---")
        for a in exs:
            print("TEXT:", a["text"][:240], ("…" if len(a["text"]) > 240 else ""))
            print("  errors:", a["errors"][cat][:3])

    for cat in ("missing", "boundary_errors", "type_errors", "spurious"):
        if summary_counts.get(cat, 0) > 0:
            _show(cat)

    print("Saved", out_path)
    print(f"[DONE] {STEP}")
except Exception as e:
    print(f"[ERROR] {STEP}: {type(e).__name__}: {e}")
    raise


## Tip chống disconnect (Colab)

- Runtime dài: ưu tiên **Colab Pro** hoặc chạy từng model (4.1 / 4.3 / 4.4) riêng.
- Bật `SYNC_TO_DRIVE=True` ở Step 4.0 để không mất `best.pt` / `results.json`.
- Có thể dán JS keep-awake trong Console (tuỳ chính sách Colab): click định kỳ vào trang.
- Tiếp tục sau reconnect: chạy lại Step **1 → 2 → 4.0**, rồi model còn thiếu; `run_multi_seed(..., skip_existing=True)` sẽ bỏ qua seed đã có `results.json`.
